In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# RetailPulse — KartZone India Pvt. Ltd.
# DELIVERIES TABLE GENERATOR 
# ═══════════════════════════════════════════════════════════════════════════════
#
# PURPOSE:
#   Generate Delivery records linked to the FINAL cleaned Orders table.
#
# SOURCE:
#   KartZone_Orders_Final.csv
#
# DATA QUALITY ISSUES INTENTIONALLY ADDED:
#   • Negative Delay_Days
#   • Missing Delivery_Partner
#   • Invalid Delivery_Status variants
#   • Missing Actual_Delivery_Days
#   • Mixed date formats
#   • Currency symbols in Delivery_Cost
#   • Inconsistent Delivery_Partner names
#   • 100 exact duplicate rows
#   • 15 completely blank rows
#   • Delivery_Attempts as string / float / null
#   • Failed deliveries with missing Failure_Reason
#   • Invalid Customer_Rating values (>5)
#   • Negative Delivery_Cost
#
# HIDDEN BUSINESS PATTERNS:
#   • Delhivery — fastest / lowest delay
#   • DTDC — slowest / highest delay
#   • Blue Dart — reliable / medium price
#   • Ekart — good for Electronics
#   • XpressBees — moderate performance
#   • Amazon Logistics — fast / reliable
#   • Oct-Dec festive period has higher delays
#   • South India has slightly longer delivery times
#   • COD orders have higher failed-delivery tendency
#   • Delayed deliveries receive lower ratings
#   • Failed deliveries receive low ratings
#
# FEATURE ENGINEERING SUPPORTED:
#   • Delivery_Delay_Days
#   • Delay_Flag
#   • On_Time_Flag
#   • Delivery_Speed_Band
#   • Is_Failed_Delivery
#   • Multiple_Attempt_Flag
#   • Cost_Per_Day
#   • Partner_Performance_Score
#   • Delay_Impact_On_Returns
# ═══════════════════════════════════════════════════════════════════════════════


import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import warnings

warnings.filterwarnings('ignore')

np.random.seed(45)
random.seed(45)


print("=" * 70)
print("  KARTZONE INDIA — DELIVERIES TABLE GENERATOR")
print("=" * 70)


# ═══════════════════════════════════════════════════════════════════════════════
# 1. REFERENCE DATA
# ═══════════════════════════════════════════════════════════════════════════════

PARTNER_VARIANTS = {
    'Delhivery': [
        'Delhivery',
        'delhivery',
        'DELHIVERY',
        'Delhivery Pvt Ltd',
        'DLV'
    ],

    'Blue Dart': [
        'Blue Dart',
        'blue dart',
        'BLUE DART',
        'BlueDart',
        'Blue-Dart'
    ],

    'DTDC': [
        'DTDC',
        'dtdc',
        'D.T.D.C.',
        'DTDC Courier',
        'Dtdc'
    ],

    'Ekart': [
        'Ekart',
        'ekart',
        'EKART',
        'eKart Logistics',
        'Ekart Pvt'
    ],

    'XpressBees': [
        'XpressBees',
        'xpressbees',
        'XPRESSBEES',
        'Xpress Bees',
        'XB'
    ],

    'Amazon Logistics': [
        'Amazon Logistics',
        'amazon logistics',
        'AMZN Logistics',
        'Amazon',
        'amzn'
    ]
}


# Partner performance profiles
PARTNER_PROFILE = {

    'Delhivery': {
        'min_delay': -1,
        'max_delay': 2,
        'base_cost': (60, 120),
        'fail_prob': 0.02
    },

    'Blue Dart': {
        'min_delay': 0,
        'max_delay': 3,
        'base_cost': (80, 160),
        'fail_prob': 0.03
    },

    'DTDC': {
    'min_delay': 0,
    'max_delay': 7,
    'base_cost': (40, 90),
    'fail_prob': 0.08
},

    'Ekart': {
        'min_delay': -1,
        'max_delay': 3,
        'base_cost': (50, 110),
        'fail_prob': 0.04
    },

    'XpressBees': {
        'min_delay': 0,
        'max_delay': 5,
        'base_cost': (45, 100),
        'fail_prob': 0.06
    },

    'Amazon Logistics': {
        'min_delay': -2,
        'max_delay': 2,
        'base_cost': (55, 115),
        'fail_prob': 0.02
    }
}


# Delhivery + Ekart dominant
PARTNER_W = [28, 18, 15, 20, 10, 9]


STATUS_VARIANTS = {

    'Delivered': [
        'Delivered',
        'delivered',
        'DELIVERED',
        'Deliverd',
        'Complete'
    ],

    'Out for Delivery': [
        'Out for Delivery',
        'out for delivery',
        'OFD',
        'Out For Del'
    ],

    'In Transit': [
        'In Transit',
        'in transit',
        'IN TRANSIT',
        'InTransit',
        'Transit'
    ],

    'Failed': [
        'Failed',
        'failed',
        'FAILED',
        'Delivery Failed',
        'Fail'
    ],

    'Returned to Origin': [
        'Returned to Origin',
        'RTO',
        'rto',
        'Return to Origin',
        'RETURNED TO ORIGIN'
    ],

    'Delayed': [
        'Delayed',
        'delayed',
        'DELAYED',
        'Behind Schedule',
        'delay'
    ]
}


WAREHOUSES = [
    'Mumbai Hub',
    'Delhi Hub',
    'Bangalore Hub',
    'Chennai Hub',
    'Hyderabad Hub',
    'Pune Hub'
]


CITY_REGION = {
    'Mumbai': 'West',
    'Delhi': 'North',
    'Bangalore': 'South',
    'Chennai': 'South',
    'Hyderabad': 'South',
    'Pune': 'West'
}


FAIL_REASONS = [
    'Customer Not Available',
    'Wrong Address',
    'Address Not Found',
    'Customer Refused Delivery',
    'Restricted Access',
    'Natural Disaster',
    None,
    None
]


MISSING = [
    None,
    np.nan,
    'NA',
    'N/A',
    '',
    'null',
    '-'
]


# ═══════════════════════════════════════════════════════════════════════════════
# 2. HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def messy_dt(dt):

    if pd.isnull(dt):
        return None

    dt = pd.Timestamp(dt).to_pydatetime()

    return dt.strftime(
        random.choice([
            '%Y-%m-%d',
            '%d/%m/%Y',
            '%d-%m-%Y',
            '%m/%d/%Y'
        ])
    )


def messy_cost(cost):

    return random.choices(
        [
            cost,
            f'₹{cost}',
            f'Rs.{cost}',
            str(cost),
            None
        ],
        weights=[55, 15, 12, 10, 8]
    )[0]


# ═══════════════════════════════════════════════════════════════════════════════
# 3. LOAD FINAL CLEANED ORDERS TABLE
# ═══════════════════════════════════════════════════════════════════════════════

print("\nLoading final cleaned Orders table...")

ORDERS_FILE = 'KartZone_Orders_Final.csv'

orders_df = pd.read_csv(ORDERS_FILE)


# Clean Order_ID
orders_df['Order_ID'] = (
    orders_df['Order_ID']
    .astype('string')
    .str.strip()
)


# Clean Order_Date
orders_df['Order_Date'] = pd.to_datetime(
    orders_df['Order_Date'],
    errors='coerce'
)


# Clean City
orders_df['City'] = (
    orders_df['City']
    .astype('string')
    .str.strip()
)


# Clean Category
orders_df['Category'] = (
    orders_df['Category']
    .astype('string')
    .str.strip()
)


# Clean Payment_Mode
orders_df['Payment_Mode'] = (
    orders_df['Payment_Mode']
    .astype('string')
    .str.strip()
)


# Remove rows without valid Order_ID or Order_Date
orders_df = orders_df.dropna(
    subset=['Order_ID', 'Order_Date']
).copy()


# Remove duplicate Order_IDs if any
orders_df = orders_df.drop_duplicates(
    subset=['Order_ID']
).reset_index(drop=True)


print(f"Valid Orders loaded : {len(orders_df):,}")
print(
    "Duplicate Order_IDs in source :",
    orders_df['Order_ID'].duplicated().sum()
)


# ═══════════════════════════════════════════════════════════════════════════════
# 4. CREATE ORDER LOOKUP
# ═══════════════════════════════════════════════════════════════════════════════

order_meta = orders_df.set_index('Order_ID').to_dict('index')

order_ids = orders_df['Order_ID'].tolist()


print(f"Orders available for Delivery generation : {len(order_ids):,}")


# ═══════════════════════════════════════════════════════════════════════════════
# 5. GENERATE DELIVERY RECORDS
# ═══════════════════════════════════════════════════════════════════════════════

print("\nGenerating one Delivery record per valid Order...")


deliveries = []


for delivery_number, oid in enumerate(order_ids, start=1):

    # -------------------------------------------------------------------------
    # Order information
    # -------------------------------------------------------------------------

    meta = order_meta[oid]

    order_date = meta['Order_Date']
    city = str(meta.get('City', '')).strip()
    category = str(meta.get('Category', '')).strip()
    payment_mode = str(meta.get('Payment_Mode', '')).strip()


    # -------------------------------------------------------------------------
    # Partner selection
    # -------------------------------------------------------------------------

    partner_key = random.choices(
        list(PARTNER_PROFILE.keys()),
        weights=PARTNER_W
    )[0]

    profile = PARTNER_PROFILE[partner_key]

    partner_val = random.choice(
        PARTNER_VARIANTS[partner_key]
    )


    # 5% missing/unassigned partner
    if random.random() < 0.05:

        partner_val = random.choice(
            MISSING[:5]
        )


    # -------------------------------------------------------------------------
    # Expected delivery days
    # -------------------------------------------------------------------------

    expected_days = random.randint(3, 7)


    # -------------------------------------------------------------------------
    # Delivery delay
    # -------------------------------------------------------------------------

    delay = random.randint(
        profile['min_delay'],
        profile['max_delay']
    )


    # -------------------------------------------------------------------------
    # South India slightly longer delivery
    # -------------------------------------------------------------------------

    if city in [
        'Bangalore',
        'Chennai',
        'Hyderabad'
    ]:

        delay += random.randint(0, 1)


    # -------------------------------------------------------------------------
    # Festive season delay
    # Oct-Nov-Dec
    # -------------------------------------------------------------------------

    if pd.notna(order_date):

        order_month = pd.Timestamp(order_date).month

        if order_month in [10, 11, 12]:

            # Festive overload
            if random.random() < 0.35:

                delay += random.randint(1, 3)


    # -------------------------------------------------------------------------
    # Actual delivery days
    # -------------------------------------------------------------------------

    actual_days = max(
        1,
        expected_days + delay
    )


    # -------------------------------------------------------------------------
    # Intentional messy Delay_Days
    # -------------------------------------------------------------------------

    delay_fmt = random.choices(

        [
            delay,
            float(delay),
            str(delay),
            None,
            -random.randint(1, 5),
            random.randint(15, 30)
        ],

        weights=[
            55,
            8,
            10,
            12,
            8,
            7
        ]
    )[0]


    # -------------------------------------------------------------------------
    # Intentional messy Actual_Delivery_Days
    # -------------------------------------------------------------------------

    actual_fmt = random.choices(

        [
            actual_days,
            float(actual_days),
            None,
            str(actual_days)
        ],

        weights=[
            65,
            8,
            17,
            10
        ]
    )[0]


    # ═════════════════════════════════════════════════════════════════════════
    # DELIVERY DATES
    # ═════════════════════════════════════════════════════════════════════════

    pickup_date = (
        pd.Timestamp(order_date) +
        pd.Timedelta(days=1)
    )


    actual_delivery_date = (
        pd.Timestamp(order_date) +
        pd.Timedelta(days=actual_days)
    )


    pickup_fmt = random.choices(

        [
            messy_dt(pickup_date),
            None,
            'NA'
        ],

        weights=[
            82,
            12,
            6
        ]
    )[0]


    delivery_fmt = random.choices(

        [
            messy_dt(actual_delivery_date),
            None,
            'NA'
        ],

        weights=[
            80,
            13,
            7
        ]
    )[0]


    # ═════════════════════════════════════════════════════════════════════════
    # DELIVERY STATUS
    # ═════════════════════════════════════════════════════════════════════════

    fail_probability = profile['fail_prob']


    # COD has higher failed-delivery probability
    if payment_mode.upper() == 'COD':

        fail_probability += 0.04


    failed = random.random() < fail_probability


    if failed:

        status_key = 'Failed'


    elif delay > 3:

        status_key = random.choices(

            [
                'Delayed',
                'In Transit',
                'Delivered',
                'Failed'
            ],

            weights=[
                40,
                30,
                20,
                10
            ]
        )[0]


    elif delay > 0:

        status_key = random.choices(

            [
                'Delivered',
                'In Transit',
                'Delayed'
            ],

            weights=[
                60,
                25,
                15
            ]
        )[0]


    elif delay < 0:

        status_key = 'Delivered'


    else:

        status_key = random.choices(

            [
                'Delivered',
                'Out for Delivery'
            ],

            weights=[
                85,
                15
            ]
        )[0]


    status_val = random.choice(
        STATUS_VARIANTS[status_key]
    )


    # ═════════════════════════════════════════════════════════════════════════
    # FAILURE REASON
    # ═════════════════════════════════════════════════════════════════════════

    failure_reason = None

    if status_key == 'Failed':

        failure_reason = random.choice(
            FAIL_REASONS
        )


    # ═════════════════════════════════════════════════════════════════════════
    # DELIVERY ATTEMPTS
    # ═════════════════════════════════════════════════════════════════════════

    if status_key == 'Failed':

        attempts = random.randint(1, 3)


    elif payment_mode.upper() == 'COD':

        # COD tends to have more attempts
        attempts = random.choices(
            [1, 2, 3],
            weights=[60, 30, 10]
        )[0]


    elif delay > 2:

        attempts = random.choices(
            [1, 2, 3],
            weights=[50, 35, 15]
        )[0]


    else:

        attempts = 1


    attempts_fmt = random.choices(

        [
            attempts,
            float(attempts),
            str(attempts),
            None
        ],

        weights=[
            65,
            10,
            15,
            10
        ]
    )[0]


    # ═════════════════════════════════════════════════════════════════════════
    # DELIVERY COST
    # ═════════════════════════════════════════════════════════════════════════

    cmin, cmax = profile['base_cost']

    cost = round(
        random.uniform(cmin, cmax),
        2
    )


    # Weight-adjusted cost
    cost = round(
        cost * random.uniform(0.8, 1.5),
        2
    )


    cost_fmt = random.choices(

        [
            messy_cost(cost),
            0,
            -cost,
            None
        ],

        weights=[
            70,
            10,
            8,
            12
        ]
    )[0]


    # ═════════════════════════════════════════════════════════════════════════
    # WAREHOUSE
    # ═════════════════════════════════════════════════════════════════════════

    warehouse = random.choice(
        WAREHOUSES
    )


    warehouse_fmt = random.choices(

        [
            warehouse,
            warehouse.lower(),
            None
        ],

        weights=[
            78,
            12,
            10
        ]
    )[0]


    # ═════════════════════════════════════════════════════════════════════════
    # CUSTOMER RATING
    # ═════════════════════════════════════════════════════════════════════════

    if status_key == 'Delivered' and delay <= 0:

        rating = random.choices(
            [4, 5, 5, 5],
            weights=[20, 30, 30, 20]
        )[0]


    elif status_key == 'Delivered' and delay <= 2:

        rating = random.choices(
            [3, 4, 5],
            weights=[30, 45, 25]
        )[0]


    elif delay > 3:

        rating = random.choices(
            [1, 2, 3],
            weights=[40, 35, 25]
        )[0]


    elif status_key == 'Failed':

        rating = random.choices(
            [1, 2],
            weights=[70, 30]
        )[0]


    else:

        rating = random.randint(1, 5)


    # Intentionally create some invalid ratings
    rating_fmt = random.choices(

        [
            rating,
            float(rating),
            None,
            random.randint(6, 10)
        ],

        weights=[
            60,
            10,
            20,
            10
        ]
    )[0]


    # ═════════════════════════════════════════════════════════════════════════
    # APPEND DELIVERY RECORD
    # ═════════════════════════════════════════════════════════════════════════

    deliveries.append({

        'Delivery_ID':
            f'DEL-{delivery_number:06d}',

        'Order_ID':
            oid,

        'Delivery_Partner':
            partner_val,

        'Warehouse_City':
            warehouse_fmt,

        'Pickup_Date':
            pickup_fmt,

        'Expected_Days':
            expected_days,

        'Actual_Delivery_Days':
            actual_fmt,

        'Delay_Days':
            delay_fmt,

        'Actual_Delivery_Date':
            delivery_fmt,

        'Delivery_Status':
            status_val,

        'Delivery_Attempts':
            attempts_fmt,

        'Failure_Reason':
            failure_reason,

        'Delivery_Cost':
            cost_fmt,

        'Customer_Rating':
            rating_fmt
    })


# ═══════════════════════════════════════════════════════════════════════════════
# 6. CREATE DATAFRAME
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.DataFrame(deliveries)


# ═══════════════════════════════════════════════════════════════════════════════
# 7. ADD INTENTIONAL DUPLICATES AND BLANK ROWS
# ═══════════════════════════════════════════════════════════════════════════════

print("\nAdding intentional duplicates and blank rows...")


# 100 exact duplicate rows
dupes = df.sample(
    100,
    random_state=45
)


# 15 completely blank rows
blanks = pd.DataFrame(
    [
        {column: None for column in df.columns}
        for _ in range(15)
    ]
)


df = pd.concat(
    [
        df,
        dupes,
        blanks
    ],
    ignore_index=True
)


# ═══════════════════════════════════════════════════════════════════════════════
# 8. SAVE RAW DELIVERY DATA
# ═══════════════════════════════════════════════════════════════════════════════

OUTPUT_FILE = 'KartZone_Deliveries.csv'

df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ═══════════════════════════════════════════════════════════════════════════════
# 9. GENERATION VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  GENERATED DELIVERY VALIDATION")
print("=" * 70)

print(f"Rows generated       : {len(df):,}")
print(f"Columns              : {len(df.columns)}")
print(f"Null values          : {df.isnull().sum().sum():,}")
print(f"Exact duplicate rows : {df.duplicated().sum()}")


print("\nOrder IDs represented:")
print(
    df['Order_ID']
    .notna()
    .sum()
)


print("\nUnique Order IDs:")
print(
    df['Order_ID']
    .nunique()
)


# ═══════════════════════════════════════════════════════════════════════════════
# 10. DATA QUALITY ISSUES EMBEDDED
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  DATA QUALITY ISSUES EMBEDDED")
print("=" * 70)

print("  ✓ Negative Delay_Days")
print("  ✓ Extreme Delay_Days outliers")
print("  ✓ Missing Actual_Delivery_Days")
print("  ✓ Missing Delivery_Partner")
print("  ✓ Inconsistent Delivery_Status")
print("  ✓ Mixed Pickup_Date formats")
print("  ✓ Mixed Actual_Delivery_Date formats")
print("  ✓ Currency symbols in Delivery_Cost")
print("  ✓ Negative Delivery_Cost")
print("  ✓ Missing Delivery_Cost")
print("  ✓ Customer_Rating values above 5")
print("  ✓ Missing Customer_Rating")
print("  ✓ Delivery_Attempts as int / float / string / null")
print("  ✓ Missing Failure_Reason")
print("  ✓ Inconsistent Delivery_Partner names")
print("  ✓ 100 duplicate delivery rows")
print("  ✓ 15 completely blank rows")


# ═══════════════════════════════════════════════════════════════════════════════
# 11. HIDDEN BUSINESS PATTERNS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  HIDDEN BUSINESS PATTERNS")
print("=" * 70)

print("  ✓ Delhivery — fastest delivery profile")
print("  ✓ DTDC — slowest delivery profile")
print("  ✓ Blue Dart — reliable / medium cost")
print("  ✓ Ekart — strong Electronics relationship")
print("  ✓ Amazon Logistics — fast / reliable")
print("  ✓ South India cities → slight delay increase")
print("  ✓ Oct-Dec → festive delivery pressure")
print("  ✓ COD → higher failed-delivery probability")
print("  ✓ COD → more delivery attempts")
print("  ✓ Failed deliveries → lower customer ratings")
print("  ✓ Delayed deliveries → lower customer ratings")


# ═══════════════════════════════════════════════════════════════════════════════
# 12. FEATURE ENGINEERING SUPPORTED
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  FEATURE ENGINEERING SUPPORTED")
print("=" * 70)

print("  ✓ Delivery_Delay_Days = Actual_Delivery_Days - Expected_Days")
print("  ✓ Delay_Flag")
print("  ✓ On_Time_Flag")
print("  ✓ Delivery_Speed_Band")
print("  ✓ Is_Failed_Delivery")
print("  ✓ Multiple_Attempt_Flag")
print("  ✓ Cost_Per_Day")
print("  ✓ Partner_Performance_Score")
print("  ✓ Delay_Impact_On_Returns")


# ═══════════════════════════════════════════════════════════════════════════════
# 13. SAMPLE DELIVERY RECORDS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  SAMPLE DELIVERY RECORDS")
print("=" * 70)

print(
    df.head(5)
)


print("\n" + "=" * 70)
print("  DELIVERY GENERATION COMPLETE")
print("=" * 70)

print(f"Saved → {OUTPUT_FILE} ✓")

  KARTZONE INDIA — DELIVERIES TABLE GENERATOR

Loading final cleaned Orders table...
Valid Orders loaded : 9,598
Duplicate Order_IDs in source : 0
Orders available for Delivery generation : 9,598

Generating one Delivery record per valid Order...

Adding intentional duplicates and blank rows...

  GENERATED DELIVERY VALIDATION
Rows generated       : 9,713
Columns              : 14
Null values          : 20,464
Exact duplicate rows : 114

Order IDs represented:
9698

Unique Order IDs:
9598

  DATA QUALITY ISSUES EMBEDDED
  ✓ Negative Delay_Days
  ✓ Extreme Delay_Days outliers
  ✓ Missing Actual_Delivery_Days
  ✓ Missing Delivery_Partner
  ✓ Inconsistent Delivery_Status
  ✓ Mixed Pickup_Date formats
  ✓ Mixed Actual_Delivery_Date formats
  ✓ Currency symbols in Delivery_Cost
  ✓ Negative Delivery_Cost
  ✓ Missing Delivery_Cost
  ✓ Customer_Rating values above 5
  ✓ Missing Customer_Rating
  ✓ Delivery_Attempts as int / float / string / null
  ✓ Missing Failure_Reason
  ✓ Inconsistent Del